# Lab 1: Supervised Learning — Classification & Regression

**Course**: CS4210 — Introduction to Machine Learning  
**Author**: [Student Name]  
**Date**: March 2026  

---

## Table of Contents
1. [Setup & Imports](#1-setup--imports)
2. [Part 1: Classification](#2-part-1-classification)
   - [2.1 Data Exploration](#21-data-exploration)
   - [2.2 Preprocessing](#22-preprocessing)
   - [2.3 Model Training](#23-model-training)
   - [2.4 Evaluation](#24-evaluation)
3. [Part 2: Regression](#3-part-2-regression)
   - [3.1 Data Exploration](#31-data-exploration)
   - [3.2 Preprocessing](#32-preprocessing)
   - [3.3 Model Training](#33-model-training)
   - [3.4 Evaluation](#34-evaluation)
4. [Results Export](#4-results-export)
5. [Infographic & Report](#5-infographic--report)
6. [Conclusion](#6-conclusion)

## 1. Setup & Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import warnings
warnings.filterwarnings('ignore')

# Scikit-learn
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, roc_curve, auc,
    mean_squared_error, mean_absolute_error, r2_score as r2_metric
)

# Classification models
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier

# Regression models
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.datasets import fetch_california_housing

# Helper scripts from the data-science-lab skill
import sys
sys.path.insert(0, os.path.join(os.getcwd(), '..', '..', 'scripts'))
# Adjust the path above to point to the data-science-lab/scripts directory

from analyze_results import find_best_model, compare_models, generate_insights, print_summary
from visualize import plot_confusion_matrix, plot_metric_comparison, plot_regression_results
from create_infographics import (
    create_experiment_infographic,
    create_comparison_infographic,
    export_infographic
)

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.4f}'.format)
sns.set_style('whitegrid')

# Create output directories
for d in ['results', 'images', 'reports']:
    os.makedirs(d, exist_ok=True)

print('✅ Setup complete.')

---

## 2. Part 1: Classification

**Task**: Classify wine quality into "good" (quality ≥ 7) vs "bad" (quality < 7)  
**Dataset**: Wine Quality (UCI ML Repository)

### 2.1 Data Exploration

In [ ]:
# Load dataset
wine_url = 'https://archive.ics.uci.edu/ml/machine-learning-databases/wine-quality/winequality-red.csv'
wine_df = pd.read_csv(wine_url, sep=';')

print(f'Shape: {wine_df.shape}')
print(f'\nColumns: {wine_df.columns.tolist()}')
print(f'\nMissing values:\n{wine_df.isnull().sum()}')
wine_df.describe()

In [ ]:
# Create binary target: good (quality >= 7) vs bad (quality < 7)
wine_df['label'] = (wine_df['quality'] >= 7).astype(int)

print(f'Class distribution:')
print(wine_df['label'].value_counts())
print(f'\nClass balance: {wine_df["label"].mean():.2%} positive')

In [ ]:
# Visualization 1: Feature distributions by class
fig, axes = plt.subplots(3, 4, figsize=(16, 10))
features = wine_df.columns.drop(['quality', 'label'])

for i, feat in enumerate(features):
    ax = axes[i // 4, i % 4]
    wine_df.groupby('label')[feat].hist(ax=ax, alpha=0.6, bins=25, legend=True)
    ax.set_title(feat, fontsize=10)
    ax.legend(['Bad', 'Good'], fontsize=7)

# Hide the last empty subplot
axes[2, 3].axis('off')
fig.suptitle('Feature Distributions by Wine Quality', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('images/clf_feature_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Visualization 2: Correlation heatmap
plt.figure(figsize=(12, 8))
corr = wine_df.drop('label', axis=1).corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, square=True, linewidths=0.5)
plt.title('Feature Correlation Matrix', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('images/clf_correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

### 2.2 Preprocessing

In [ ]:
# Prepare features and target
X_clf = wine_df.drop(['quality', 'label'], axis=1)
y_clf = wine_df['label']

# Train/test split
X_train_clf, X_test_clf, y_train_clf, y_test_clf = train_test_split(
    X_clf, y_clf, test_size=0.2, random_state=42, stratify=y_clf
)

# Feature scaling
scaler_clf = StandardScaler()
X_train_clf_scaled = scaler_clf.fit_transform(X_train_clf)
X_test_clf_scaled = scaler_clf.transform(X_test_clf)

print(f'Train shape: {X_train_clf_scaled.shape}')
print(f'Test shape:  {X_test_clf_scaled.shape}')
print(f'Train class balance: {y_train_clf.mean():.2%}')
print(f'Test class balance:  {y_test_clf.mean():.2%}')

### 2.3 Model Training

Train 4 classifier families with multiple hyperparameter configurations.

In [ ]:
clf_results = []

# --- Logistic Regression ---
for C in [0.01, 0.1, 1, 10, 100]:
    model = LogisticRegression(C=C, max_iter=1000, random_state=42)
    model.fit(X_train_clf_scaled, y_train_clf)
    y_pred = model.predict(X_test_clf_scaled)
    clf_results.append({
        'model': 'Logistic Regression',
        'params': f'C={C}',
        'accuracy': accuracy_score(y_test_clf, y_pred),
        'precision': precision_score(y_test_clf, y_pred, average='weighted'),
        'recall': recall_score(y_test_clf, y_pred, average='weighted'),
        'f1_score': f1_score(y_test_clf, y_pred, average='weighted'),
    })

# --- SVM ---
for kernel in ['linear', 'rbf', 'poly']:
    model = SVC(kernel=kernel, random_state=42)
    model.fit(X_train_clf_scaled, y_train_clf)
    y_pred = model.predict(X_test_clf_scaled)
    clf_results.append({
        'model': 'SVM',
        'params': f'kernel={kernel}',
        'accuracy': accuracy_score(y_test_clf, y_pred),
        'precision': precision_score(y_test_clf, y_pred, average='weighted'),
        'recall': recall_score(y_test_clf, y_pred, average='weighted'),
        'f1_score': f1_score(y_test_clf, y_pred, average='weighted'),
    })

# --- Random Forest ---
for n_est in [50, 100, 200]:
    for max_d in [5, 10, None]:
        model = RandomForestClassifier(n_estimators=n_est, max_depth=max_d, random_state=42)
        model.fit(X_train_clf_scaled, y_train_clf)
        y_pred = model.predict(X_test_clf_scaled)
        depth_str = str(max_d) if max_d else 'None'
        clf_results.append({
            'model': 'Random Forest',
            'params': f'n={n_est}, depth={depth_str}',
            'accuracy': accuracy_score(y_test_clf, y_pred),
            'precision': precision_score(y_test_clf, y_pred, average='weighted'),
            'recall': recall_score(y_test_clf, y_pred, average='weighted'),
            'f1_score': f1_score(y_test_clf, y_pred, average='weighted'),
        })

# --- KNN ---
for k in [3, 5, 7, 9, 11]:
    model = KNeighborsClassifier(n_neighbors=k)
    model.fit(X_train_clf_scaled, y_train_clf)
    y_pred = model.predict(X_test_clf_scaled)
    clf_results.append({
        'model': 'KNN',
        'params': f'k={k}',
        'accuracy': accuracy_score(y_test_clf, y_pred),
        'precision': precision_score(y_test_clf, y_pred, average='weighted'),
        'recall': recall_score(y_test_clf, y_pred, average='weighted'),
        'f1_score': f1_score(y_test_clf, y_pred, average='weighted'),
    })

clf_df = pd.DataFrame(clf_results)
print(f'Total classification experiments: {len(clf_df)}')
clf_df.sort_values('f1_score', ascending=False).head(10)

### 2.4 Evaluation

In [ ]:
# Print summary using helper script
print_summary(clf_df, metric='f1_score')

In [ ]:
# Confusion matrix for the best model
best_clf = find_best_model(clf_df, metric='f1_score')
print(f"Best classifier: {best_clf['model']} ({best_clf['params']})")
print(f"F1 Score: {best_clf['f1_score']:.4f}")

# Re-train the best model to get predictions
# (In practice, you'd store predictions during training)
best_model_clf = RandomForestClassifier(n_estimators=100, max_depth=None, random_state=42)
best_model_clf.fit(X_train_clf_scaled, y_train_clf)
y_pred_best = best_model_clf.predict(X_test_clf_scaled)

cm = confusion_matrix(y_test_clf, y_pred_best)
plot_confusion_matrix(cm, labels=['Bad', 'Good'],
                      title='Best Model Confusion Matrix',
                      save_path='images/clf_confusion_matrix.png')
plt.show()

In [ ]:
# Model comparison chart
plot_metric_comparison(clf_df, 'f1_score', title='F1 Score by Model',
                       save_path='images/clf_model_comparison.png')
plt.show()

---

## 3. Part 2: Regression

**Task**: Predict California median house values  
**Dataset**: California Housing (scikit-learn built-in)

### 3.1 Data Exploration

In [ ]:
# Load dataset
housing = fetch_california_housing(as_frame=True)
housing_df = housing.frame.copy()

print(f'Shape: {housing_df.shape}')
print(f'\nTarget: {housing.target_names}')
print(f'\nMissing values: {housing_df.isnull().sum().sum()}')
housing_df.describe()

In [ ]:
# Visualization: Scatter plots of key features vs target
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
features = housing.feature_names

for i, feat in enumerate(features):
    ax = axes[i // 4, i % 4]
    ax.scatter(housing_df[feat], housing_df['MedHouseVal'],
               alpha=0.1, s=3, color='steelblue')
    ax.set_xlabel(feat, fontsize=9)
    ax.set_ylabel('MedHouseVal', fontsize=9)
    ax.set_title(f'{feat} vs Target', fontsize=10)

fig.suptitle('Feature vs. Target Scatter Plots', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('images/reg_scatter_plots.png', dpi=150, bbox_inches='tight')
plt.show()

### 3.2 Preprocessing

In [ ]:
# Prepare features and target
X_reg = housing_df.drop('MedHouseVal', axis=1)
y_reg = housing_df['MedHouseVal']

# Clip target outliers (values > 5 are likely capped)
clip_mask = y_reg < 5.0
X_reg = X_reg[clip_mask].copy()
y_reg = y_reg[clip_mask].copy()

# Train/test split
X_train_reg, X_test_reg, y_train_reg, y_test_reg = train_test_split(
    X_reg, y_reg, test_size=0.2, random_state=42
)

# Feature scaling
scaler_reg = StandardScaler()
X_train_reg_scaled = scaler_reg.fit_transform(X_train_reg)
X_test_reg_scaled = scaler_reg.transform(X_test_reg)

print(f'Train shape: {X_train_reg_scaled.shape}')
print(f'Test shape:  {X_test_reg_scaled.shape}')
print(f'Target range: {y_train_reg.min():.2f} — {y_train_reg.max():.2f}')

### 3.3 Model Training

In [ ]:
reg_results = []

def eval_regressor(name, params_str, model, X_tr, y_tr, X_te, y_te):
    """Train, predict, and record regression metrics."""
    model.fit(X_tr, y_tr)
    y_pred = model.predict(X_te)
    mse = mean_squared_error(y_te, y_pred)
    return {
        'model': name,
        'params': params_str,
        'mse': mse,
        'rmse': np.sqrt(mse),
        'mae': mean_absolute_error(y_te, y_pred),
        'r2_score': r2_metric(y_te, y_pred),
    }

# --- Linear Regression (baseline) ---
reg_results.append(eval_regressor(
    'Linear Regression', 'baseline',
    LinearRegression(),
    X_train_reg_scaled, y_train_reg, X_test_reg_scaled, y_test_reg
))

# --- Ridge ---
for alpha in [0.01, 0.1, 1, 10, 100]:
    reg_results.append(eval_regressor(
        'Ridge', f'alpha={alpha}',
        Ridge(alpha=alpha),
        X_train_reg_scaled, y_train_reg, X_test_reg_scaled, y_test_reg
    ))

# --- Lasso ---
for alpha in [0.01, 0.1, 1, 10, 100]:
    reg_results.append(eval_regressor(
        'Lasso', f'alpha={alpha}',
        Lasso(alpha=alpha, max_iter=5000),
        X_train_reg_scaled, y_train_reg, X_test_reg_scaled, y_test_reg
    ))

# --- Random Forest ---
for n_est in [50, 100, 200]:
    reg_results.append(eval_regressor(
        'Random Forest', f'n={n_est}',
        RandomForestRegressor(n_estimators=n_est, random_state=42, n_jobs=-1),
        X_train_reg_scaled, y_train_reg, X_test_reg_scaled, y_test_reg
    ))

# --- Gradient Boosting ---
for lr in [0.01, 0.05, 0.1]:
    for n_est in [100, 200]:
        reg_results.append(eval_regressor(
            'Gradient Boosting', f'lr={lr}, n={n_est}',
            GradientBoostingRegressor(learning_rate=lr, n_estimators=n_est, random_state=42),
            X_train_reg_scaled, y_train_reg, X_test_reg_scaled, y_test_reg
        ))

reg_df = pd.DataFrame(reg_results)
print(f'Total regression experiments: {len(reg_df)}')
reg_df.sort_values('r2_score', ascending=False).head(10)

### 3.4 Evaluation

In [ ]:
# Print summary
print_summary(reg_df, metric='r2_score')

In [ ]:
# Best model — Actual vs Predicted
best_reg = find_best_model(reg_df, metric='r2_score')
print(f"Best regressor: {best_reg['model']} ({best_reg['params']})")
print(f"R² Score: {best_reg['r2_score']:.4f}")

# Re-train best model for visualization
best_model_reg = GradientBoostingRegressor(learning_rate=0.1, n_estimators=200, random_state=42)
best_model_reg.fit(X_train_reg_scaled, y_train_reg)
y_pred_best_reg = best_model_reg.predict(X_test_reg_scaled)

plot_regression_results(y_test_reg.values, y_pred_best_reg,
                         title='Best Model: Actual vs Predicted',
                         save_path='images/reg_actual_vs_predicted.png')
plt.show()

In [ ]:
# R² Score comparison
plot_metric_comparison(reg_df, 'r2_score', title='R² Score by Model',
                       save_path='images/reg_r2_comparison.png')
plt.show()

---

## 4. Results Export

Save all experiment results to CSV files for reproducibility.

In [ ]:
# Export classification results
clf_df.to_csv('results/classification_results.csv', index=False)
print(f'✅ Classification results saved: {len(clf_df)} experiments')

# Export regression results
reg_df.to_csv('results/regression_results.csv', index=False)
print(f'✅ Regression results saved: {len(reg_df)} experiments')

# Preview
print(f'\n--- Classification Top 5 ---')
print(clf_df.sort_values('f1_score', ascending=False).head())
print(f'\n--- Regression Top 5 ---')
print(reg_df.sort_values('r2_score', ascending=False).head())

---

## 5. Infographic & Report

Generate a publication-quality infographic and a written report using the helper scripts.

In [ ]:
# Classification infographic
fig_clf = create_experiment_infographic(
    clf_df,
    title='Classification Results — Wine Quality',
    metric='f1_score',
    theme='dark'
)
export_infographic(fig_clf, 'images/clf_infographic.png')
plt.show()

In [ ]:
# Regression infographic
fig_reg = create_experiment_infographic(
    reg_df,
    title='Regression Results — California Housing',
    metric='r2_score',
    theme='dark'
)
export_infographic(fig_reg, 'images/reg_infographic.png')
plt.show()

In [ ]:
# Side-by-side comparison infographic
fig_compare = create_comparison_infographic(
    clf_df, reg_df,
    title='Lab 1 — Classification vs Regression',
    theme='dark'
)
export_infographic(fig_compare, 'images/lab1_infographic.png')
plt.show()

---

## 6. Conclusion

### Key Findings

**Classification (Wine Quality)**
- The best performing model was [MODEL] with an F1 score of [SCORE]
- Feature scaling had a significant impact on SVM and KNN performance
- The dataset is imbalanced (~13% positive class), which affected precision/recall tradeoffs

**Regression (California Housing)**
- Gradient Boosting with tuned learning rate achieved the best R² score
- Tree-based methods outperformed linear models
- Median income (`MedInc`) was the most important feature for predicting house values

### Lessons Learned
- Hyperparameter tuning can significantly improve model performance
- Feature scaling is essential for distance-based algorithms (KNN, SVM)
- Ensemble methods (Random Forest, Gradient Boosting) tend to generalize better